# BERT en español sobre los títulos · `ap-2026-q-2`
Afina (*fine-tuning*) un modelo de lenguaje preentrenado para predecir `used` a partir del **título** (y la garantía), con los **mismos 5 folds** que el notebook principal. Guarda una predicción *fuera de fold* (OOF) para el train y una para el test, que el principal usa como una variable más (`BERT_DIR`).

**Por qué puede ayudar:** el texto es tu bloque más valioso (quitar `text_oof` costaba −0,0036), pero el TF-IDF solo ve palabras sueltas. Un BERT entiende contexto ("sin uso" vs "uso"), variantes y errores de tipeo.

**Requisitos en Kaggle (panel derecho → Session options)**
1. **Accelerator: GPU** (T4 o P100). En CPU sería inviable y el notebook se detiene.
2. **Internet: On**, para descargar el modelo de Hugging Face (o agregá el modelo como *Input* y poné su ruta en `MODEL_NAME`).
3. **Add Input:** la competencia `ap-2026-q-2`.

**Cómo usarlo**
1. Primero `DEBUG_ONE_FOLD = True`: entrena **un solo fold** (calculo ~5-10 min en GPU; es una estimación mía). Compará su accuracy con las referencias de abajo.
2. Si es prometedor, `DEBUG_ONE_FOLD = False` y corré los 5 folds. Cada fold se guarda al terminar: si la sesión se corta, volvé a correr y **retoma** donde quedó.
3. **Save Version → Save & Run All**, y en el notebook principal: *Add Input → Notebook Output* y `BERT_DIR = '/kaggle/input/NOMBRE-DE-ESTE-NOTEBOOK'`.

**Variantes que conviene probar (cada una es una corrida aparte; el principal acepta varias en `BERT_EXTRA_DIRS`):** `USE_META_TEXT = True` (el modelo ve también categoría, fotos, precio y envío), otro `MODEL_NAME` (p. ej. `PlanTL-GOB-ES/roberta-base-bne`), y `EPOCHS = 3`.

**Referencias de accuracy solo-texto (de tu notebook principal):** TF-IDF del título `title_w` ≈ 0,82 · todas las fuentes de texto juntas `all` ≈ 0,87. Si el BERT (título + garantía) queda claramente por encima de 0,82, aporta algo nuevo.

## 1. Configuración

In [1]:
# ======================= CONFIGURACIÓN =======================
# Modelo (los nombres son los que recuerdo de Hugging Face; si alguno falla, probá otro o usá una ruta local en /kaggle/input/...)
# BETO (español). Alternativas: 'distilbert-base-multilingual-cased' (más rápido), 'xlm-roberta-base', 'PlanTL-GOB-ES/roberta-base-bne'
MODEL_NAME = 'PeterPanecillo/PlanTL-GOB-ES-roberta-base-bne-copy'
MODEL_REVISION = '38ec9f9192e36a3f66ded3145a1b7c7ac585a169'
USE_WARRANTY = True       # agrega el texto de la garantía como segundo segmento
USE_CATEGORY = False      # True = antepone la categoría al título (el modelo puede aprender significados por categoría)
USE_META_TEXT = False     # True = además ve, como texto: categoría, tamaño de foto, cantidad de fotos, tramo de precio (global y dentro de la categoría), envío y tipo de listado. Sin usar el target
MAX_LEN      = 64         # largo máximo en tokens (los títulos son cortos). Con USE_META_TEXT se sube solo a 128
EPOCHS = 3
LR = 3e-5
BATCH = 32
WARMUP = 0.06
WEIGHT_DECAY = 0.01
SEEDS        = [0]        # [0, 1] promedia 2 modelos por fold (el doble de lento, algo más estable)
DEBUG_ONE_FOLD = False     # True = solo el fold 0 (para probar). Después ponelo en False
ALLOW_CPU    = False      # solo para depurar con un modelo diminuto; con un BERT real es inviable
SAMPLE_TRAIN = None       # debe coincidir con el notebook principal (None = todo el train)
OUT_DIR      = '.'

TARGET = 'condition'; RANDOM_STATE = 42; N_FOLDS = 5          # igual que el notebook principal (mismos folds)
DATA_DIR = None; DOWNLOAD_WITH_KAGGLEHUB = True

## 2. Datos
Carga `train` y `test`, define el target (`used` = 1) y los folds, igual que el notebook principal (así las predicciones OOF son compatibles).

In [2]:
import os, re, time, hashlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import torch, torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)

tok = RobertaTokenizerFast.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION
)

print("Tokenizador:", type(tok).__name__)
print("Vocabulario:", tok.vocab_size)
def list_files():
    dirs = [DATA_DIR] if DATA_DIR else ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub/competitions')]
    return [os.path.join(r, f) for d in dirs if d and os.path.isdir(d) for r, _, fs in os.walk(d) for f in fs if f.endswith('.jsonlines')]
paths = list_files()
if not paths and DOWNLOAD_WITH_KAGGLEHUB:
    import kagglehub
    base = kagglehub.competition_download('ap-2026-q-2'); paths = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs if f.endswith('.jsonlines')]
pick = lambda w: [p for p in paths if w in os.path.basename(p).lower()][0]
train = pd.read_json(pick('train'), lines=True); test = pd.read_json(pick('test'), lines=True)
if SAMPLE_TRAIN: train = train.sample(SAMPLE_TRAIN, random_state=RANDOM_STATE)
train = train.reset_index(drop=True); test = test.reset_index(drop=True); ntr, nte = len(train), len(test)
if USE_META_TEXT and MAX_LEN < 128: MAX_LEN = 128; print('USE_META_TEXT: MAX_LEN subido a 128')
classes = sorted(train[TARGET].unique()); y = (train[TARGET] == classes[-1]).astype(int).values     # 'used' = 1
RUN_SIG = hashlib.md5(str((
    "roberta_embedding_fix_v1",
    MODEL_NAME,
    MODEL_REVISION,
    USE_WARRANTY, USE_CATEGORY, USE_META_TEXT,
    MAX_LEN, EPOCHS, BATCH, LR, WARMUP,
    WEIGHT_DECAY, ntr, nte, RANDOM_STATE, N_FOLDS
)).encode()).hexdigest()[:10]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'train {train.shape} | test {test.shape} | % {classes[-1]}: {y.mean():.3f} | dispositivo: {device} | firma de la corrida: {RUN_SIG}')
if device == 'cpu' and not ALLOW_CPU:
    raise RuntimeError('No hay GPU. Activá Accelerator = GPU en Session options (o ALLOW_CPU=True solo para depurar).')

tokenizer_config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/851k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/509k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

Tokenizador: RobertaTokenizer
Vocabulario: 50262
train (70000, 45) | test (30000, 44) | % used: 0.463 | dispositivo: cuda | firma de la corrida: 75a45a6dd1


## 3. Texto → tokens
Título (+ garantía como segundo segmento). Se usa el texto **original** (con mayúsculas y acentos), porque el modelo es *cased*.

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
def tokenizer_from_bpe_files(name):
    # RoBERTa (p. ej. el español de la BNE) usa BPE a nivel de bytes: si la detección automática falla, se arma el tokenizador a mano con vocab.json + merges.txt
    import json
    from tokenizers import Tokenizer, models, pre_tokenizers, decoders, processors
    from transformers import PreTrainedTokenizerFast
    if os.path.isdir(name): vocab_f, merges_f = os.path.join(name, 'vocab.json'), os.path.join(name, 'merges.txt')
    else:
        from huggingface_hub import hf_hub_download
        vocab_f, merges_f = hf_hub_download(name, 'vocab.json'), hf_hub_download(name, 'merges.txt')
    vocab = json.load(open(vocab_f, encoding='utf-8'))
    tk = Tokenizer(models.BPE.from_file(vocab_f, merges_f))
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False); tk.decoder = decoders.ByteLevel()
    tk.post_processor = processors.TemplateProcessing(single='<s> $A </s>', pair='<s> $A </s> </s> $B </s>', special_tokens=[('<s>', vocab['<s>']), ('</s>', vocab['</s>'])])
    kw = {k: v for k, v in dict(bos_token='<s>', eos_token='</s>', sep_token='</s>', cls_token='<s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>').items() if v in vocab}
    return PreTrainedTokenizerFast(tokenizer_object=tk, model_max_length=512, model_input_names=['input_ids', 'attention_mask'], **kw)

def load_tokenizer(name):
    try:
        tok = RobertaTokenizerFast.from_pretrained(name)
        print('Tokenizador RoBERTa cargado correctamente')
        return tok
    except Exception as e:
        print('Falló RobertaTokenizerFast:', str(e)[:300])
        print('Intentando BPE manual...')
        return tokenizer_from_bpe_files(name)

tok = load_tokenizer(MODEL_NAME)

clean = lambda s: s.fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

def meta_strings(df_all):
    # texto con datos estructurados de cada publicación (train + test juntos; el tramo de precio se calcula con ambos, sin usar el target)
    first = df_all['pictures'].map(lambda l: l[0] if isinstance(l, list) and l and isinstance(l[0], dict) else {})
    size = first.map(lambda p: p.get('size') or 'sin foto'); msize = first.map(lambda p: p.get('max_size') or '')
    npic = df_all['pictures'].map(lambda l: len(l) if isinstance(l, list) else 0)
    price = np.log1p(pd.to_numeric(df_all['price'], errors='coerce').clip(lower=0).fillna(0))
    decil = pd.qcut(price.rank(method='first'), 10, labels=False)
    rel = np.minimum((price.groupby(df_all['category_id']).rank(pct=True) * 5).astype(int), 4)      # tramo de precio dentro de su categoría
    ship = df_all['shipping'].map(lambda d: 'envío gratis' if isinstance(d, dict) and d.get('free_shipping') else 'envío pago')
    return ('categoría ' + df_all['category_id'].astype(str) + ' | foto ' + size + ' ' + msize + ' | fotos ' + npic.astype(str) + ' | precio ' + decil.astype(str)
            + ' | precio en categoría ' + rel.astype(str) + ' | ' + ship + ' | ' + df_all['listing_type_id'].astype(str)).str.replace(r'\s+', ' ', regex=True).tolist()
META = meta_strings(pd.concat([train, test], ignore_index=True)) if USE_META_TEXT else None

def encode(df, meta=None):
    a = clean(df['title'])
    if USE_CATEGORY: a = df['category_id'].astype(str) + ' : ' + a
    b = clean(df['warranty']).str.slice(0, 120) if USE_WARRANTY else None
    if meta is not None:
        m = pd.Series(meta, index=df.index); b = (b + ' | ' + m) if b is not None else m
    if b is not None: enc = tok(a.tolist(), b.tolist(), truncation=True, max_length=MAX_LEN)
    else: enc = tok(a.tolist(), truncation=True, max_length=MAX_LEN)
    keys = list(enc.keys()); return [{k: enc[k][i] for k in keys} for i in range(len(a))]
t0 = time.time(); feats_tr, feats_te = encode(train, META[:ntr] if META else None), encode(test, META[ntr:] if META else None)
lens = np.array([len(f['input_ids']) for f in feats_tr])
print(f'tokenizado en {time.time()-t0:.0f}s | largo en tokens: mediana {np.median(lens):.0f}, p95 {np.percentile(lens, 95):.0f}, máx {lens.max()} | truncados: {(lens >= MAX_LEN).mean():.2%}')
print('ejemplo:', tok.decode(feats_tr[0]['input_ids']))

Tokenizador RoBERTa cargado correctamente
tokenizado en 6s | largo en tokens: mediana 19, p95 40, máx 64 | truncados: 0.09%
ejemplo: <s>Pizarron 20x30 + Imagen + Bolsita Artesanal- Yk42artesanias</s></s>Calidad y mejor precio</s>


## 4. Entrenamiento y predicción
Precisión mixta (fp16) en GPU, AdamW con calentamiento y decaimiento lineal. **Se usa la última época** (no se elige la mejor mirando la validación), para que el OOF no salga optimista.

In [4]:
collate = lambda batch: tok.pad(batch, return_tensors='pt')
use_amp = device == 'cuda'
try: make_scaler = lambda: torch.amp.GradScaler('cuda', enabled=use_amp)
except Exception: make_scaler = lambda: torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def predict(model, feats, idx):
    model.eval(); idx = np.asarray(idx); order = sorted(range(len(idx)), key=lambda i: len(feats[idx[i]]['input_ids'])); out = np.zeros(len(idx))
    for s in range(0, len(order), 256):
        ii = order[s:s + 256]; batch = {k: v.to(device) for k, v in collate([feats[idx[i]] for i in ii]).items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits
        out[ii] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return out

from transformers import (
    RobertaForMaskedLM,
    RobertaForSequenceClassification
)

def train_fold(tr_i, va_i, seed):
    import copy
    torch.manual_seed(seed)
    np.random.seed(seed)

    # 1. Cargar el modelo original de lenguaje
    mlm = RobertaForMaskedLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        use_safetensors=False
    )

    # 2. Crear el clasificador binario
    config = mlm.config
    config.num_labels = 2

    model = RobertaForSequenceClassification(config)

    # 3. Transferir TODOS los pesos de RoBERTa
    model.roberta.load_state_dict(
        mlm.roberta.state_dict(),
        strict=True
    )

    del mlm

    # 4. Ajustar vocabulario si el tokenizador tiene tokens adicionales
    if len(tok) != model.config.vocab_size:
        model.resize_token_embeddings(len(tok))

    model = model.to(device)

    print("Pesos de RoBERTa transferidos correctamente")

    # Desde acá continúa tu entrenamiento original:
    nd = ('bias', 'LayerNorm.weight')

    # ... resto de tu función sin cambios
    opt = torch.optim.AdamW([{'params': [p for n, p in model.named_parameters() if not any(x in n for x in nd)], 'weight_decay': WEIGHT_DECAY},
                             {'params': [p for n, p in model.named_parameters() if any(x in n for x in nd)], 'weight_decay': 0.0}], lr=LR)
    steps = EPOCHS * int(np.ceil(len(tr_i) / BATCH)); sch = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps); scaler = make_scaler()
    best_acc = -1
    best_state = None
    best_epoch = 0
    for ep in range(EPOCHS):
        model.train(); perm = np.random.permutation(tr_i); tot = 0.0; t0 = time.time()
        for s in range(0, len(perm), BATCH):
            idx = perm[s:s + BATCH]; batch = {k: v.to(device) for k, v in collate([feats_tr[i] for i in idx]).items()}
            labels = torch.from_numpy(y[idx]).long().to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(**batch).logits.float(), labels)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item() * len(idx)
        pv = predict(model, feats_tr, va_i)
        val_acc = ((pv > .5) == y[va_i]).mean()
        
        print(
            f'   época {ep + 1}/{EPOCHS} | '
            f'loss train {tot / len(perm):.4f} | '
            f'acc val {val_acc:.4f} | {time.time() - t0:.0f}s'
        )
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = ep + 1
    model.load_state_dict(best_state)
    
    print(f'Mejor época: {best_epoch} | mejor accuracy: {best_acc:.4f}')
    
    return model

In [5]:
config = RobertaConfig.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION
)

print("Arquitectura:", config.model_type)
print("Vocabulario modelo:", config.vocab_size)
print("Vocabulario tokenizador:", tok.vocab_size)
print("Hidden size:", config.hidden_size)

assert config.model_type == "roberta"
assert config.vocab_size == tok.vocab_size

print("Configuración compatible.")

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Arquitectura: roberta
Vocabulario modelo: 50262
Vocabulario tokenizador: 50262
Hidden size: 768
Configuración compatible.


In [6]:
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(ntr); pred_te = np.zeros(nte); done = np.zeros(ntr, bool); n_folds = 0
for k, (tr_i, va_i) in enumerate(skf.split(np.zeros(ntr), y)):
    if DEBUG_ONE_FOLD and k > 0: break
    print(f'--- fold {k} ---'); pv_sum = np.zeros(len(va_i)); pt_sum = np.zeros(nte)
    for seed in SEEDS:
        f = os.path.join(OUT_DIR, f'bert_fold{k}_seed{seed}_{RUN_SIG}.npz')
        if os.path.exists(f):
            z = np.load(f); pv, pt = z['pv'], z['pt']; print(f'   (retomado de {f})')
        else:
            t0 = time.time(); model = train_fold(tr_i, va_i, seed)
            pv = predict(model, feats_tr, va_i); pt = predict(model, feats_te, np.arange(nte)); np.savez(f, pv=pv, pt=pt)
            del model; torch.cuda.empty_cache(); print(f'   fold {k} semilla {seed}: {time.time() - t0:.0f}s')
        pv_sum += pv / len(SEEDS); pt_sum += pt / len(SEEDS)
    oof[va_i] = pv_sum; pred_te += pt_sum; done[va_i] = True; n_folds += 1
    print(f'fold {k}: acc = {((oof[va_i] > .5) == y[va_i]).mean():.4f}')
pred_te /= max(n_folds, 1)
print(f'\nAccuracy OOF del BERT (solo texto): {((oof[done] > .5) == y[done]).mean():.4f}   (referencias: title_w ≈ 0,82 · all ≈ 0,87)')

--- fold 0 ---


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Pesos de RoBERTa transferidos correctamente
   época 1/3 | loss train 0.4098 | acc val 0.8382 | 204s
   época 2/3 | loss train 0.2439 | acc val 0.8557 | 205s
   época 3/3 | loss train 0.0917 | acc val 0.8469 | 204s
Mejor época: 2 | mejor accuracy: 0.8557
   fold 0 semilla 0: 643s
fold 0: acc = 0.8557
--- fold 1 ---


... (8 outputs left out)

Pesos de RoBERTa transferidos correctamente
   época 1/3 | loss train 0.4094 | acc val 0.8423 | 205s
   época 2/3 | loss train 0.2394 | acc val 0.8583 | 205s
   época 3/3 | loss train 0.0914 | acc val 0.8522 | 205s
Mejor época: 2 | mejor accuracy: 0.8583
   fold 3 semilla 0: 633s
fold 3: acc = 0.8583
--- fold 4 ---


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Pesos de RoBERTa transferidos correctamente
   época 1/3 | loss train 0.4095 | acc val 0.8507 | 205s
   época 2/3 | loss train 0.2470 | acc val 0.8579 | 205s
   época 3/3 | loss train 0.0957 | acc val 0.8558 | 205s
Mejor época: 2 | mejor accuracy: 0.8579
   fold 4 semilla 0: 632s
fold 4: acc = 0.8579

Accuracy OOF del BERT (solo texto): 0.8554   (referencias: title_w ≈ 0,82 · all ≈ 0,87)


## 5. Guardar resultados
`bert_feats_train.csv` (predicción fuera de fold + el target, para verificar que coincide con el principal), `bert_feats_test.csv` y un `submission_bert.csv` (solo para referencia: por sí solo no va a superar a tu mezcla).

In [7]:
if DEBUG_ONE_FOLD:
    print('⚠ DEBUG_ONE_FOLD = True: solo hay un fold. No guardo archivos para el notebook principal (pondrían ceros en los otros folds).')
else:
    pd.DataFrame({'bert_oof': oof, 'y_check': y}).to_csv(os.path.join(OUT_DIR, 'bert_feats_train.csv'), index=False)
    pd.DataFrame({'bert_oof': pred_te}).to_csv(os.path.join(OUT_DIR, 'bert_feats_test.csv'), index=False)
    pd.DataFrame({'ID': range(1, nte + 1), TARGET: [classes[i] for i in (pred_te > .5).astype(int)]}).to_csv(os.path.join(OUT_DIR, 'submission_bert.csv'), index=False)
    print('guardado: bert_feats_train.csv, bert_feats_test.csv, submission_bert.csv')

guardado: bert_feats_train.csv, bert_feats_test.csv, submission_bert.csv


## 6. Cómo usarlo en el notebook principal
En su celda de configuración:
```python
BERT_DIR        = '/kaggle/input/NOMBRE-DE-ESTE-NOTEBOOK'   # carpeta con bert_feats_train.csv y bert_feats_test.csv
BERT_AS_FEATURE = True     # suma 'bert_oof' como variable de LightGBM / XGBoost / CatBoost (lo más probable que ayude)
BERT_IN_BLEND   = False    # True = además lo suma como modelo en la mezcla final
```
Corré el principal completo (se reentrenan los modelos porque cambian las features) y compará el CV de la mezcla contra el que tenías. Si querés medir cuánto aporta: `RUN_ABLATION = True` incluye el bloque `bert`.